# Phase 4 walkthrough — SQL storage

Everything the pipeline produces is loaded into a SQL database through SQLAlchemy (SQLite by default; set
`DB_URL` to point at PostgreSQL or SQL Server). This notebook lists the tables, runs the portable example queries
in `sql/example_queries.sql`, and checks a few numbers against the Phase 2/3 reports. The dashboard
(`streamlit run src/dashboard.py`) reads only from this database.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from sqlalchemy import inspect, text, func, select
from src import db
from src.config import load_config
cfg = load_config()
eng = db.get_engine(cfg)
if not inspect(eng).has_table("run"):
    from src.run import phase4; phase4(cfg)
print("database:", db.safe_url(db.db_url(cfg)))
run_id = db.latest_run_id(eng); print("latest run_id:", run_id)

database: sqlite:////home/user/aeso_probabilistic_congestion_assessment/data/aeso_poc.sqlite
latest run_id: 1


## Tables and row counts

In [2]:
with eng.connect() as c:
    counts = {t.name: c.execute(select(func.count()).select_from(t)).scalar() for t in db.metadata.sorted_tables}
pd.Series(counts, name="rows").to_frame()

,rows
bus,6
run,1
generator,13
line,8
lmp_quantile,102
runtime,6
scenario,37520
surrogate_metric,252
bus_result,225120
gen_dispatch,487760


## Portable example queries (`sql/example_queries.sql`)

In [3]:
sql = Path("sql/example_queries.sql").read_text()
blocks = [b.strip() for b in sql.split(";") if "SELECT" in b]
params = {"run_id": run_id, "true_value": True}
with eng.connect() as c:
    for b in blocks:
        title = [l for l in b.splitlines() if l.startswith("-- ")][-1].removeprefix("-- ")
        display(pd.read_sql(text(b), c, params=params).head(12).style.set_caption(title).format(precision=4))

,line_id,p_binding,n_scenarios
0,SOUTH-CAL,0.2231,20000
1,BC-CAL,0.0704,20000
2,EDM-NORTH-1,0.0316,20000
3,EDM-NORTH-2,0.0303,20000
4,SOUTH-RD,0.0059,20000
5,CAL-EDM,0.0029,20000
6,RD-EDM,0.0001,20000
7,CAL-RD,0.0000,20000


,bus_id,topology,mean_lmp,n
0,BC,all lines in,48.1771,18355
1,BC,line(s) out,50.2479,1645
2,CAL,all lines in,49.4413,18355
3,CAL,line(s) out,54.4430,1645
4,EDM,all lines in,47.5252,18355
5,EDM,line(s) out,51.4874,1645
6,NORTH,all lines in,47.4427,18355
7,NORTH,line(s) out,42.2584,1645
8,RD,all lines in,46.5671,18355
9,RD,line(s) out,51.5049,1645


,model,bus_id,mae_congested,n
0,LightGBM,BC,1.2757,6617
1,LightGBM,CAL,2.3936,6617
2,LightGBM,EDM,2.0956,6617
3,LightGBM,NORTH,1.5818,6617
4,LightGBM,RD,2.1413,6617
5,LightGBM,SOUTH,2.9901,6617
6,MLP,BC,1.6084,6617
7,MLP,CAL,2.8382,6617
8,MLP,EDM,2.5005,6617
9,MLP,NORTH,2.3838,6617


,month,hours_at_limit,hours
0,1,270,1488
1,2,335,1344
2,3,341,1488
3,4,318,1440
4,5,305,1488
5,6,206,1440
6,7,151,1488
7,8,217,1488
8,9,411,1440
9,10,352,1488


,outaged_line,share_hours_overload,max_post_loading,worst_line,delta_cost_musd_per_year,rank_security,rank_economic
0,EDM-NORTH-1,0.9942,2.0000,EDM-NORTH-2,154.0456,1,1
1,EDM-NORTH-2,0.9942,2.0000,EDM-NORTH-1,154.0456,1,1
2,CAL-RD,0.6408,1.4224,SOUTH-CAL,25.1122,3,5
3,RD-EDM,0.5463,1.7527,CAL-EDM,15.3446,4,6
4,SOUTH-CAL,0.3982,2.4103,SOUTH-RD,140.4880,5,3
5,SOUTH-RD,0.3220,1.6571,SOUTH-CAL,72.4081,6,4
6,CAL-EDM,0.2226,1.1000,SOUTH-CAL,-2.3733,7,8
7,BC-CAL,nan,nan,nan,5.3863,8,7


## Same statistic, three ways
The Monte Carlo congestion probability for SOUTH-CAL, computed (1) from the raw `line_result` rows in SQL,
(2) from the stored summary table, (3) from the Phase 2 CSV report. They must agree.

In [4]:
with eng.connect() as c:
    q1 = pd.read_sql(text(blocks[0]), c, params=params).set_index("line_id").loc["SOUTH-CAL", "p_binding"]
    q2 = pd.read_sql(text("SELECT p_binding FROM line_congestion_stat WHERE run_id=:r AND source='mc_opf' AND line_id='SOUTH-CAL'"), c, params={"r": run_id}).iloc[0, 0]
q3 = pd.read_csv("reports/tables/mc_line_congestion.csv", index_col=0).loc["SOUTH-CAL", "p_congested"]
print(q1, q2, q3); assert abs(q1 - q2) < 1e-12 and abs(q2 - q3) < 1e-12

0.2231 0.2231 0.2231


## Schema (generated DDL excerpt for SQL Server)

In [5]:
print(Path("sql/schema_mssql.sql").read_text().split("CREATE TABLE line_result")[1].split(";")[0])

 (
	run_id INTEGER NOT NULL, 
	study VARCHAR(32) NOT NULL, 
	scenario_id INTEGER NOT NULL, 
	line_id VARCHAR(64) NOT NULL, 
	flow_mw FLOAT NULL, 
	loading FLOAT NULL, 
	binding BIT NULL, 
	PRIMARY KEY (run_id, study, scenario_id, line_id), 
	FOREIGN KEY(run_id, study, scenario_id) REFERENCES scenario (run_id, study, scenario_id), 
	FOREIGN KEY(line_id) REFERENCES line (line_id)
)
